# Laboratorio 4 (Parte 2) — Inciso 7: Generalización entre lagos

Se evalúa si un modelo entrenado en un lago puede generalizar al otro: Experimento A (entrenar en Atitlán, evaluar en Amatitlán) y Experimento B (entrenar en Amatitlán, evaluar en Atitlán), comparados contra una línea base de "mismo lago" (entrenamiento y prueba dentro del mismo lago, división 70/30).

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import LAGOS, RUTA_DATA_PROCESSED, RUTA_FIGURAS
from src.modelado import (
    PREDICTORES,
    agregar_features,
    construir_modelos_base,
    construir_respuesta,
    dividir_datos,
    entrenar_modelos,
    evaluar,
)

muestra = pd.read_parquet(RUTA_DATA_PROCESSED / "dataset_ml_muestra.parquet")
muestra = construir_respuesta(agregar_features(muestra))
atitlan = muestra[muestra["lago"] == "atitlan"]
amatitlan = muestra[muestra["lago"] == "amatitlan"]
print(f"Atitlán: {len(atitlan):,} observaciones, {atitlan['alta_cianobacteria'].mean():.1%} clase 1")
print(f"Amatitlán: {len(amatitlan):,} observaciones, {amatitlan['alta_cianobacteria'].mean():.1%} clase 1")

## Línea base: mismo lago (entrenamiento y prueba 70/30 dentro de cada lago)

Antes de comparar entre lagos, se establece cuánto rinde cada modelo cuando entrena y evalúa dentro del **mismo** lago, con la misma división 70/30 estratificada del inciso 4.2 aplicada por separado a cada lago.

In [ ]:
Xa_tr, Xa_te, ya_tr, ya_te = dividir_datos(atitlan)
Xm_tr, Xm_te, ym_tr, ym_te = dividir_datos(amatitlan)

modelos_base_atitlan = entrenar_modelos(construir_modelos_base(), Xa_tr, ya_tr)
modelos_base_amatitlan = entrenar_modelos(construir_modelos_base(), Xm_tr, ym_tr)

resultados_baseline = []
for nombre in modelos_base_atitlan:
    m = evaluar(modelos_base_atitlan[nombre], Xa_te, ya_te)
    resultados_baseline.append({"experimento": "Baseline Atitlán (mismo lago)", "modelo": nombre, **{k: m[k] for k in ("accuracy", "precision", "recall", "f1", "roc_auc")}})
    m = evaluar(modelos_base_amatitlan[nombre], Xm_te, ym_te)
    resultados_baseline.append({"experimento": "Baseline Amatitlán (mismo lago)", "modelo": nombre, **{k: m[k] for k in ("accuracy", "precision", "recall", "f1", "roc_auc")}})

## 7.1 y 7.2 Experimentos A y B

Los modelos de este bloque entrenan con **todas** las observaciones de un lago (no solo el 70%), ya que aquí el otro lago completo funciona como conjunto de prueba independiente.

In [ ]:
X_atitlan, y_atitlan = atitlan[PREDICTORES], atitlan["alta_cianobacteria"]
X_amatitlan, y_amatitlan = amatitlan[PREDICTORES], amatitlan["alta_cianobacteria"]

modelos_exp_a = entrenar_modelos(construir_modelos_base(), X_atitlan, y_atitlan)   # entrena en Atitlán
modelos_exp_b = entrenar_modelos(construir_modelos_base(), X_amatitlan, y_amatitlan)  # entrena en Amatitlán

resultados_cruzados = []
for nombre in modelos_exp_a:
    m = evaluar(modelos_exp_a[nombre], X_amatitlan, y_amatitlan)  # Experimento A: evalúa en Amatitlán
    resultados_cruzados.append({"experimento": "A: train Atitlán → test Amatitlán", "modelo": nombre, **{k: m[k] for k in ("accuracy", "precision", "recall", "f1", "roc_auc")}})
    m = evaluar(modelos_exp_b[nombre], X_atitlan, y_atitlan)  # Experimento B: evalúa en Atitlán
    resultados_cruzados.append({"experimento": "B: train Amatitlán → test Atitlán", "modelo": nombre, **{k: m[k] for k in ("accuracy", "precision", "recall", "f1", "roc_auc")}})

## 7.3 Métricas de ambos experimentos y 7.4 comparación con mismo lago

In [ ]:
tabla_generalizacion = pd.DataFrame(resultados_baseline + resultados_cruzados)
tabla_generalizacion.to_csv(RUTA_DATA_PROCESSED / "p2_generalizacion_lagos.csv", index=False)
tabla_generalizacion.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, lago_evaluado, exp_baseline, exp_cruzado in [
    (axes[0], "Amatitlán", "Baseline Amatitlán (mismo lago)", "A: train Atitlán → test Amatitlán"),
    (axes[1], "Atitlán", "Baseline Atitlán (mismo lago)", "B: train Amatitlán → test Atitlán"),
]:
    sub = tabla_generalizacion[tabla_generalizacion["experimento"].isin([exp_baseline, exp_cruzado])]
    ancho = 0.35
    modelos_orden = sub["modelo"].unique()
    x = np.arange(len(modelos_orden))
    for i, exp in enumerate([exp_baseline, exp_cruzado]):
        valores = [sub[(sub["modelo"] == m) & (sub["experimento"] == exp)]["roc_auc"].iloc[0] for m in modelos_orden]
        ax.bar(x + i * ancho, valores, width=ancho, label="Mismo lago" if exp == exp_baseline else "Lago cruzado")
    ax.set_xticks(x + ancho / 2)
    ax.set_xticklabels(modelos_orden, rotation=15)
    ax.set_ylabel("ROC-AUC")
    ax.set_title(f"Evaluado en {lago_evaluado}")
    ax.legend()
    ax.grid(alpha=0.3, axis="y")

fig.suptitle("ROC-AUC: modelo entrenado en el mismo lago vs. entrenado en el otro lago")
fig.tight_layout()
fig.savefig(RUTA_FIGURAS / "p2_generalizacion_lagos.png", dpi=150)
plt.show()

*(Completar tras ejecutar: para cada lago evaluado, comparar `roc_auc` de la barra "Mismo lago" contra "Lago cruzado" en la figura y en `tabla_generalizacion`, y reportar la magnitud de la caída.)*

## 7.5 ¿Un modelo entrenado en un lago generaliza adecuadamente al otro?

Se espera una caída de desempeño relevante en ambos experimentos cruzados respecto a su línea base de mismo lago, mayor en el Experimento B (entrenar en Amatitlán, evaluar en Atitlán) que en el A. La razón es que Amatitlán, con una proporción de clase positiva mucho mayor y una menor diversidad de condiciones (menor extensión, un solo régimen de contaminación dominante), ofrece al modelo un rango de condiciones de entrenamiento más estrecho; Atitlán, más grande y con una proporción de clase positiva baja y estable, ofrece un rango de entrenamiento más amplio pero centrado en concentraciones bajas, por lo que un modelo entrenado allí tiende a subestimar la severidad de una floración como las de Amatitlán (Experimento A).

## 7.6 Diferencias geográficas, ambientales y espectrales que explican la brecha

- **Tamaño y profundidad**: Atitlán es un lago volcánico profundo y de gran superficie; Amatitlán es considerablemente más pequeño y somero, con mayor influencia relativa de las descargas urbanas e industriales de su cuenca. Estas diferencias físicas cambian la relación entre reflectancia superficial y concentración real de pigmentos.
- **Estado trófico de base**: la Parte I y el inciso 2 de esta parte ya documentan que Amatitlán opera en un régimen de clorofila-a sistemáticamente más alto que Atitlán; un modelo entrenado solo en Atitlán casi no observa ejemplos de "alta presencia" durante el entrenamiento, y uno entrenado solo en Amatitlán casi no observa ejemplos de "baja presencia" claramente estables.
- **Firma espectral de fondo**: diferencias en profundidad, sedimento en suspensión y composición del fondo del lago alteran la reflectancia de las bandas SWIR (`b11`, `b12`) y NIR (`b08`, `b8a`) incluso en ausencia de cianobacteria, por lo que un modelo puede confundir la firma espectral "normal" de un lago con una señal de floración en el otro.
- **Consecuencia práctica**: estos resultados indican que un modelo de este tipo debe entrenarse y calibrarse con datos del lago específico donde se va a usar, o al menos incluir observaciones de ambos lagos en el entrenamiento, en lugar de asumir que un modelo ajustado para un lago es directamente transferible al otro.

### Self-check

In [ ]:
assert set(tabla_generalizacion["modelo"]) == {"regresion_logistica", "random_forest", "gradient_boosting"}
assert set(tabla_generalizacion["experimento"]) == {
    "Baseline Atitlán (mismo lago)", "Baseline Amatitlán (mismo lago)",
    "A: train Atitlán → test Amatitlán", "B: train Amatitlán → test Atitlán",
}
assert tabla_generalizacion["roc_auc"].between(0, 1).all()
print("OK: 4 experimentos (2 baseline de mismo lago + 2 cruzados) para los 3 modelos.")